# প্রাধান্য সদস্য মिडলওয়্যার সহ হোটেল বুকিং

এই নোটবুকটি Microsoft Agent Framework ব্যবহার করে **ফাংশন-ভিত্তিক মিডলওয়্যার** প্রদর্শন করে। আমরা শর্তাধীন ওয়ার্কফ্লো উদাহরণের উপর ভিত্তি করে একটি মিডলওয়্যার স্তর যোগ করি যা প্রাধান্য সদস্যদের বিশেষ সুবিধা দেয়।

## আপনি যা শিখবেন:
১. **ফাংশন-ভিত্তিক মিডলওয়্যার**: ফাংশনের ফলাফল ইন্টারসেপ্ট এবং পরিবর্তন করা
২. **প্রসঙ্গ অ্যাক্সেস**: কার্যকরির পরে `context.result` পড়া এবং পরিবর্তন করা
৩. **ব্যবসায়িক লজিক বাস্তবায়ন**: প্রাধান্য সদস্য সুবিধাসমূহ
৪. **ফলাফল ওভাররাইড**: ব্যবহারকারীর স্থিতির উপর ভিত্তি করে ফাংশন ফলাফল পরিবর্তন
৫. **একই ওয়ার্কফ্লো, ভিন্ন ফলাফল**: মিডলওয়্যার-চালিত আচরণ পরিবর্তন

## মিডলওয়্যার সহ ওয়ার্কফ্লো আর্কিটেকচার:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## শর্তাধীন ওয়ার্কফ্লো থেকে মূল পার্থক্য:

**মিডলওয়্যার ছাড়া** (14-conditional-workflow.ipynb):
- প্যারিসে কোনও রুম নেই → alternative_agent এ পাঠানো

**মিডলওয়্যার সহ** (এই নোটবুক):
- নিয়মিত ব্যবহারকারী + প্যারিস → রুম নেই → alternative_agent এ পাঠানো
- প্রাধান্য ব্যবহারকারী + প্যারিস → 🌟 মিডলওয়্যার ওভাররাইড করে! → উপলব্ধ → booking_agent এ পাঠানো

## পূর্বশর্তসমূহ:
- Microsoft Agent Framework ইনস্টল করা
- শর্তাধীন ওয়ার্কফ্লো বোঝাপড়া (14-conditional-workflow.ipynb দেখুন)
- GitHub টোকেন বা OpenAI API কী
- মিডলওয়্যার প্যাটার্নের মৌলিক ধারণা


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ধাপ ১: কাঠামোবদ্ধ আউটপুটের জন্য Pydantic মডেলগুলি সংজ্ঞায়িত করুন

এই মডেলগুলি সেই **স্কিমা** নির্ধারণ করে যা এজেন্টরা ফেরত দেবে। আমরা একটি `priority_override` ক্ষেত্র যোগ করেছি যখন মিডলওয়্যার উপলব্ধতার ফলাফল পরিবর্তন করে তা ট্র্যাক করার জন্য। 


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ধাপ ২: প্রায়োরিটি সদস্যের ডাটাবেস সংজ্ঞায়িত করা

এই ডেমোর জন্য, আমরা একটি প্রায়োরিটি সদস্যপদ ডাটাবেস সিমুলেট করব। প্রোডাকশনে, এটি একটি বাস্তব ডাটাবেস বা API থেকে তথ্য নেবে।

**প্রায়োরিটি সদস্যগণ:**
- `alice@example.com` - ভিআইপি সদস্য
- `bob@example.com` - প্রিমিয়াম সদস্য  
- `priority_user` - টেস্ট অ্যাকাউন্ট


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## ধাপ ৩: হোটেল বুকিং সরঞ্জাম তৈরি করা

শর্তাধীন কর্মপ্রবাহের মতোই, কিন্তু এবার এটি মিডলওয়্যার দ্বারা আটকানো হবে!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ধাপ ৪: 🌟 অগ্রাধিকার পরীক্ষা মিডলওয়্যার তৈরি করুন (মূল বৈশিষ্ট্য!)

এটি এই নোটবুকের **মূল কার্যকারিতা**। মিডলওয়্যারটি:

১. **হোটেল_বুকিং** ফাংশন কলকে আটকায়
২. `next(context)` কল করে ফাংশনটি সাধারণভাবে চালায়
৩. `context.result` এ ফলাফল পরীক্ষা করে
৪. যদি ব্যবহারকারী অগ্রাধিকারপ্রাপ্ত হয় এবং কোনো কক্ষ না পাওয়া যায় তবে ফলাফলটি ওভাররাইড করে
৫. পরিবর্তিত ফলাফলটি এজেন্টের কাছে ফেরত পাঠায়

**মূল প্যাটার্ন:**
```python
async def my_middleware(context, next):
    await next(context)  # ফাংশন কার্যকর করুন
    # এখন context.result এ ফাংশনের আউটপুট রয়েছে
    if some_condition:
        context.result = new_value  # ওভাররাইড!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## ধাপ ৫: রাউটিংয়ের জন্য শর্তাধীন ফাংশনগুলি সংজ্ঞায়িত করা

শর্তাধীন ওয়ার্কফ্লোর মত একই শর্তাধীন ফাংশনগুলি - তারা রাউটিং নির্ধারণের জন্য গঠনবদ্ধ আউটপুট পরিদর্শন করে।


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## ধাপ ৬: কাস্টম ডিসপ্লে এক্সিকিউটার তৈরি করুন

পূর্বের মতো একই এক্সিকিউটার - চূড়ান্ত ওয়ার্কফ্লো আউটপুট প্রদর্শন করে।


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## ধাপ ৭: পরিবেশ ভেরিয়েবল সংরক্ষণ করুন

LLM ক্লায়েন্ট (Microsoft Foundry / Azure OpenAI) কনফিগার করুন।


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ধাপ ৮: মিডলওয়্যার সহ এআই এজেন্ট তৈরি করুন

**মূল পার্থক্য:** যখন availability_agent তৈরি করা হয়, আমরা `middleware` প্যারামিটার প্রেরণ করি!

এজেন্টের ফাংশন আহ্বান পাইপলাইনে priority_check_middleware কীভাবে ইনজেক্ট করা হয় তা এইভাবেই করি।


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ধাপ ৯: ওয়ার্কফ্লো তৈরি করুন

আগের মতো একই ওয়ার্কফ্লো কাঠামো - প্রাপ্যতার ভিত্তিতে শর্তসাপেক্ষ রুটিং। 


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## পর্যায় ১০: পরীক্ষার কেস ১ - প্যারিসে সাধারণ ব্যবহারকারী (অভাররাইট নেই)

একজন সাধারণ ব্যবহারকারী প্যারিস বুক করার চেষ্টা করে → কোন রুম নেই → alternate_agent এ রুট করা হয়


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## ধাপ ১১: টেস্ট কেস ২ - 🌟 প্যারিসের অগ্রাধিকার ব্যবহারকারী (ওভাররাইড সহ!)

একটি অগ্রাধিকার সদস্য প্যারিসে বুক করার চেষ্টা করে → প্রথমে কোন রুম নেই → 🌟 মিডলওয়্যার ওভাররাইড করে! → বুকিং এজেন্টের কাছে রুট করা হয়

**এটাই মিডলওয়্যারের ক্ষমতার মূল প্রদর্শনী!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## ধাপ ১২: টেস্ট কেস ৩ - স্টকহোমে অগ্রাধিকার ব্যবহারকারী (ইতিমধ্যে উপলব্ধ)

অগ্রাধিকার ব্যবহারকারী স্টকহোম চেষ্টা করে → কক্ষ উপলব্ধ → ওভাররাইডের প্রয়োজন নেই → booking_agent এ রুট করে

এটি দেখায় যে মধ্যবর্তী সফটওয়্যার দরকার হলে তবেই কাজ করে!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## মূল বিষয়বস্তু এবং মিডলওয়্যার ধারণাগুলো

### ✅ যা আপনি শিখেছেন:

#### **১. ফাংশন-ভিত্তিক মিডলওয়্যার প্যাটার্ন**

মিডলওয়্যার একটি সরল অ্যাসিঙ্ক ফাংশন ব্যবহার করে ফাংশন কল ধরতে পারে:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ফাংশন কার্যকর করার আগে
    print("Intercepting...")
    
    # ফাংশনটি কার্যকর করুন
    await next(context)
    
    # ফাংশন কার্যকর করার পরে - ফলাফল পরীক্ষা করুন
    if context.result:
        # প্রয়োজনে ফলাফল পরিবর্তন করুন
        context.result = modified_value
```

#### **২. প্রসঙ্গ অ্যাক্সেস এবং ফলাফল ওভাররাইড**

- `context.function` - যে ফাংশনটি কল হচ্ছে সেটিতে অ্যাক্সেস
- `context.arguments` - ফাংশনের আর্গুমেন্ট পড়া
- `context.kwargs` - অতিরিক্ত প্যারামিটার অ্যাক্সেস
- `await next(context)` - ফাংশনটি কার্যকর করা
- `context.result` - ফাংশনের আউটপুট পড়া/পরিবর্তন

#### **৩. ব্যবসায়িক লজিক বাস্তবায়ন**

আমাদের মিডলওয়্যার অগ্রাধিকার সদস্য সুবিধা বাস্তবায়ন করে:
- **নিয়মিত ব্যবহারকারী**: কোনো পরিবর্তন নয়, সাধারণ ওয়ার্কফ্লো
- **অগ্রাধিকার ব্যবহারকারী**: "উপলভ্যতা নেই" → "উপলভ্য" ওভাররাইড
- **শর্তাধীন লজিক**: শুধুমাত্র প্রয়োজন হলে ওভাররাইড

#### **৪. একই ওয়ার্কফ্লো, ভিন্ন ফলাফল**

মিডলওয়্যারের ক্ষমতা:
- ✅ ওয়ার্কফ্লো স্ট্রাকচারে কোনো পরিবর্তন নেই
- ✅ টুল ফাংশনে কোনো পরিবর্তন নেই
- ✅ শর্তাধীন রাউটিং লজিকে কোনো পরিবর্তন নেই
- ✅ শুধু মিডলওয়্যার → ভিন্ন আচরণ!

### 🚀 বাস্তব জীবনের প্রয়োগসমূহ:

১. **ভিআইপি/প্রিমিয়াম ফিচার**
   - প্রিমিয়াম ব্যবহারকারীদের জন্য রেট লিমিট ওভাররাইড
   - সম্পদের অগ্রাধিকার অ্যাক্সেস প্রদান
   - প্রিমিয়াম ফিচার গতিশীলভাবে আনলক করা

২. **এ/বি টেস্টিং**
   - বিভিন্ন ব্যবহারকারীদের ভিন্ন বাস্তবায়নে পাঠানো
   - নির্দিষ্ট ব্যবহারকারীদের সঙ্গে নতুন ফিচার পরীক্ষা
   - ধাপে ধাপে ফিচার রোলআউট

৩. **নির্বাচন ও অনুসরণ**
   - ফাংশন কলগুলি অডিট করা
   - সংবেদনশীল অপারেশন ব্লক করা
   - ব্যবসায়িক নিয়ম প্রয়োগ

৪. **দক্ষতা উন্নতি**
   - নির্দিষ্ট ব্যবহারকারীদের জন্য ফলাফল ক্যাশ করা
   - সম্ভব হলে ব্যয়বহুল অপারেশন এড়ানো
   - গতিশীল সম্পদ বরাদ্দ

৫. **ত্রুটি হ্যান্ডলিং ও রিট্রাই**
   - ত্রুটি ধরা এবং সুচিন্তিতভাবে পরিচালনা করা
   - রিট্রাই লজিক বাস্তবায়ন করা
   - বিকল্প বাস্তবায়নে ফিরে যাওয়া

৬. **লগিং ও মোনিটরিং**
   - ফাংশন কার্যকররণের সময় ট্র্যাক করা
   - প্যারামিটার এবং ফলাফল লগ করা
   - ব্যবহার প্যাটার্ন পর্যবেক্ষণ করা

### 🔑 ডেকোরেটর থেকে মূল পার্থক্যগুলো:

| বৈশিষ্ট্য | ডেকোরেটর | মিডলওয়্যার |
|---------|-----------|------------|
| **সীমা** | একক ফাংশন | এজেন্টের সমস্ত ফাংশন |
| **নমনীয়তা** | নির্দিষ্ট সংজ্ঞায় | রানটাইমে গতিশীল |
| **প্রসঙ্গ** | সীমিত | পূর্ণ এজেন্ট প্রসঙ্গ |
| **কম্পোজিশন** | একাধিক ডেকোরেটর | মিডলওয়্যার পাইপলাইন |
| **এজেন্ট সচেতন** | না | হ্যাঁ (এজেন্ট অবস্থা অ্যাক্সেস) |

### 📚 কখন মিডলওয়্যার ব্যবহার করবেন:

✅ **মিডলওয়্যার ব্যবহার করুন যখন:**
- ব্যবহারকারী/সেশন অবস্থা অনুযায়ী আচরণ পরিবর্তন দরকার
- একাধিক ফাংশনে লজিক প্রয়োগ করতে চান
- এজেন্ট-স্তরের প্রসঙ্গ অ্যাক্সেস দরকার
- আপনি ক্রস-কাটিং বিষয় (লগিং, অথেন্টিকেশন, ইত্যাদি) বাস্তবায়ন করছেন

❌ **মিডলওয়্যার ব্যবহার করবেন না যখন:**
- সরল ইনপুট ভ্যালিডেশন প্রয়োজন (Pydantic ব্যবহার করুন)
- ফাংশন-নির্দিষ্ট লজিক (ফাংশনে রাখুন)
- এককালীন পরিবর্তন (সরাসরি ফাংশন পরিবর্তন করুন)

### 🎓 উন্নত প্যাটার্নসমূহ:

```python
# একাধিক মিডলওয়্যার (কার্যকারিতা ক্রম গুরুত্বপূর্ণ!)
middleware=[
    logging_middleware,      # প্রথম লগ করে
    auth_middleware,         # তারপর অথ পরীক্ষা করে
    cache_middleware,        # তারপর ক্যাশ পরীক্ষা করে
    rate_limit_middleware,   # তারপর রেট সীমা নির্ধারণ করে
    priority_check_middleware  # অবশেষে অগ্রাধিকার পরীক্ষা করে
]

# শর্তসাপেক্ষ মিডলওয়্যার কার্যকরী
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ফলাফল পরিবর্তন করে
    else:
        # সম্পূর্ণ কার্যকরী অতিক্রম করে
        context.result = cached_value
```

### 🔗 সংশ্লিষ্ট ধারণাসমূহ:

- **এজেন্ট মিডলওয়্যার**: এজেন্ট.run() কলগুলো ধরবে
- **ফাংশন মিডলওয়্যার**: টুল ফাংশন কলগুলো ধরবে (আমরা যা ব্যবহার করেছি!)
- **মিডলওয়্যার পাইপলাইন**: ক্রমাগত মিডলওয়্যার এক্সিকিউশন চেইন
- **প্রসঙ্গ প্রোপাগেশন**: মিডলওয়্যার চেইনের মাধ্যমে অবস্থা প্রেরণ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
